# Quickstart: a label list, a zero-shot guess, a head-only fine-tune

`ds.model(labels)` gives you Laya with your labels. It answers zero-shot right away, and `.train()` fine-tunes only
its decision head on your rows, which runs on a laptop CPU or Apple GPU in a few minutes. The held-out rows below use
phrasings that never appear in training, so the "after" number is an honest estimate on this synthetic set.

Runs on: a laptop (CPU or MPS). No API key. First run downloads the Laya English checkpoint (about 850 MB).

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations. decisionsmith is an independent project, not affiliated with TypeSafe AI or Convai Innovations.

In [1]:
# uv pip install "decisionsmith[laya]"
import decisionsmith as ds

## 1. Synthetic tickets
Three teams, template sentences. Swap in your own `(text, label)` rows.

In [2]:
import random

TEAMS = {
    "billing": [
        "I was charged twice this month", "my invoice shows the wrong amount", "the tax on my bill looks wrong",
        "there is a fee on my statement I don't recognise", "I need a copy of last month's invoice",
        "my card was charged after I cancelled", "the payment failed but the money left my account",
        "why did my bill go up", "please update the billing address on my invoices", "I was billed for two seats",
    ],
    "technical": [
        "the dashboard shows a 500 error", "I can't log in since the update", "exports are stuck at 99 percent",
        "password reset emails never arrive", "uploads time out after a minute", "the app crashes when I open reports",
        "sync between devices stopped working", "the API returns 401 with a valid key", "search results are empty",
        "the page is blank in Safari",
    ],
    "sales": [
        "can I get a quote for 50 seats", "do you offer discounts for nonprofits", "is there an annual pricing option",
        "what's the difference between pro and team", "I want to upgrade my plan", "can we talk about volume pricing",
        "do you have an enterprise plan", "is there a free trial for teams", "can I pay by invoice for a yearly plan",
        "who do I talk to about a partnership",
    ],
}
OPENERS = ["", "Hi, ", "Hello team, ", "Good morning. ", "Hey, "]
CLOSERS = ["", " Thanks.", " Any update?", " This is urgent.", " Can you help?", " Please look into it."]


def make(cores, n, seed):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        team = rng.choice(sorted(cores))
        core = rng.choice(cores[team])
        text = rng.choice(OPENERS) + core + "." + rng.choice(CLOSERS)
        rows.append((text[0].upper() + text[1:], team))
    return rows


# held-out test sentences never appear in training: the last 3 phrasings of every team
train_rows = make({t: xs[:-3] for t, xs in TEAMS.items()}, 300, seed=0)
test_rows = make({t: xs[-3:] for t, xs in TEAMS.items()}, 90, seed=1)
print(len(train_rows), "training rows,", len(test_rows), "held-out rows, e.g.", train_rows[0])

300 training rows, 90 held-out rows, e.g. ('Good morning. do you have an enterprise plan.', 'sales')


## 2. Zero-shot
No training yet: Laya reads the labels and the question and picks one.

In [3]:
model = ds.model(["billing", "technical", "sales"], question="Which team should handle this ticket?")
test_texts = [t for t, _ in test_rows]
truth = [label for _, label in test_rows]


def accuracy(preds):
    return sum(p == y for p, y in zip(preds, truth)) / len(truth)


before = model.predict(test_texts)
print("zero-shot accuracy on held-out rows: %.2f" % accuracy(before))
for text, guess in list(zip(test_texts, before))[:5]:
    print("%-9s <- %s" % (guess, text))

<venv>/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<venv>/lib/python3.12/site-packages/laya/agent.py:928: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


zero-shot accuracy on held-out rows: 0.92
billing   <- I was billed for two seats. Any update?
billing   <- Good morning. please update the billing address on my invoices. This is urgent.
technical <- Hi, search results are empty.
sales     <- Good morning. is there a free trial for teams. This is urgent.
technical <- Good morning. the API returns 401 with a valid key. Any update?


## 3. Fine-tune the head
`train()` holds out part of the rows for calibration and testing, trains, and switches to the new weights unless they came out worse.

In [4]:
report = model.train(train_rows, out="runs/quickstart")
print(report)

trained on 225 rows · accuracy 0.82 -> 0.93 on 45 held-out decisions · now using runs/quickstart
  note: only 45 held-out decisions, so these numbers are rough; more data helps
finetune: laya -> runs/quickstart

field  test  base_acc  accuracy  macro_f1  base_ece  ece  
-----  ----  --------  --------  --------  --------  -----
label  45    0.822     0.933     0.930     0.162     0.106
all    45    0.822     0.933     0.930     0.162     0.106

go: no
  - test split has 45 decisions; want 100 (add data)
  - calibration error 0.106 is above 0.10
saved: ./runs/quickstart


In [5]:
after = model.predict(test_texts)
print("held-out accuracy: %.2f zero-shot -> %.2f fine-tuned" % (accuracy(before), accuracy(after)))

held-out accuracy: 0.92 zero-shot -> 0.92 fine-tuned


## 4. Save and load
The folder is a plain Laya checkpoint: `laya.load(path)` works too. If training came out worse, `train()` kept the old model and there is nothing new to save.

In [6]:
if model.trained is None:
    print("training came out worse, so the old model was kept and there is nothing new to save; add rows, train again")
else:
    path = model.save("my-ticket-model")  # my-ticket-model-v1, then -v2: versions are never overwritten
    again = ds.load(path)
    print(again.predict("My card was charged after I cancelled."))

billing


Next: put it behind an LLM with `ds.harness(model, teacher=...)` ([03_harness_shadow_to_cascade.ipynb](03_harness_shadow_to_cascade.ipynb)).